# **Q: Given client and campaign information available at the time of contact, predict whether the client subscribes to a term deposit (i.e., a type of short-term investment).**

**Import Libraries**

# **1. Identifying the Prediction Target**

The business objective of the campaign is to determine, before or during a call, which clients are likely to subscribe to a term deposit. This allows the bank to prioritise outreach to high-probability clients, reducing campaign costs, and to improve conversion rates by targeting the right customer segments.

The variable y directly encodes the outcome of interest, the target variable (for this assignment) is y. It is a binary variable that answers the question: "Has the client subscribed to a term deposit?" which means whether contact resulted in a subscription with possible values of (yes) or (no).

It satisfies every criterion for a valid prediction target:
- **Causal role:** it is the effect we want to predict, not a cause.
- **Tmporality**: it is observed after the campaign contact, making it a valid label for supervised learning.
- **Business-relevance:** predicting it enables actionable decisions before a call is made.
- **Clean and unambiguous:** a binary outcome with no sentinel values or encoding quirks. In the language of Lecture 1, this is a binary classification problem: we map a feature vector x (client and campaign attributes available at the time of contact) to a label y ∈ {no, yes}.
- **Measurability:** It is a clean, unambiguous binary outcome with no sentinel values. Accodring to Lecture 1, this is a **binary classification problem**: we map a feature vector $\mathbf{x}$ (client and campaign attributes available at the time of contact) to a label $y \in \{\text{no}, \text{yes}\}$.


#### **Variables that could superficially appear to be valid targets**

Several variables in the dataset might initially seem like reasonable prediction targets but are not appropriate for this task.

1. `duration`: records how long the previous phone call lasted (in seconds).

It is strongly correlated with y. A call that ends quickly likely means the client declined, while a long call often leads to a subscription. One might be tempted to treat duration as a proxy or surrogate for y. However, duration is not known before the call is made. At prediction time, when we want to decide whether to call a client at all, this value simply does not yet exist. The dataset documentation explicitly warns that this variable should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model. Using duration as a target, or even as a feature, constitutes target leakage: it carries information that is only available after the outcome has effectively been determined.

2. `poutcome`: records whether a prior campaign resulted in a success, failure, or nonexistent outcome (meaning no prior contact).

It is strongly predictive of y and might appear to be a valid target because it also captures campaign success. However, poutcome refers to a past event from a previous, different campaign, not the current one. It is historical information that is already known at prediction time. Treating it as the target would mean predicting something already observed, rather than future subscription behaviour, making the task trivial and commercially useless. It is also a three-class categorical variable, which does not map cleanly onto the binary classification objective defined by the bank.

3. `campaign` : number of contacts during this campaign

This counts how many times the client was contacted during the current campaign. It might seem like a target because it reflects the intensity of the bank's effort — and one could imagine wanting to predict "how many calls will it take?". However:
It's a campaign management variable, not a client outcome. The bank controls it; the client doesn't decide it.
It is partially observed during the campaign, meaning it accumulates as contacts happen — it's not a clean post-hoc label.
Predicting campaign would reframe the problem as a regression task about operational effort, completely ignoring whether the client actually subscribes.
Most critically, it carries no information about the success of the campaign from the client's perspective.

4. `pdays` : number of days since last contact from a previous campaign; 999 = never contacted

This might look like a target because 999 is a sentinel value that encodes a meaningful state ("never contacted before"), giving it a quasi-categorical feel. Someone might think: "predict whether this client is a new prospect or a returning one."
But:
pdays is historical metadata — it is fully known at the time of the call and describes the past, not the future.
Predicting it would mean predicting something the bank already knows before dialling. It has zero predictive utility as a target.
The 999 sentinel also makes it statistically unusual (heavily right-skewed with a mass point), which could make it look like it needs special treatment as a label — but this is a red herring. It's an input feature quirk, not a labelling problem.
The business has no interest in "predicting" how many days ago a client was last called — that's a lookup, not a prediction.


**Summary:**

y is the only column that satisfies all three requirements for a valid prediction target. It is the desired business outcome, it is not available at prediction time, and it is free of post-hoc information. Duration, poutcome, campaign, and pdays each fail at least one of these criteria, for reasons rooted in leakage, temporal ordering, or a fundamental mismatch with the business objective.


# **2. Data Loading and Exploration**

This is the next logical step because everything else depends on actually understanding what's in the data. You can't make informed decisions about splitting, encoding, missing values, or scaling without first inspecting the dataset's structure, distributions etc.

In [6]:
#load libraries

import pandas as pd
import numpy as np


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


**2.1 Load dataset into pandas dataframe**

In [22]:
data = pd.read_csv('bank-additional.csv',sep = ';')
data

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4114,30,admin.,married,basic.6y,no,yes,yes,cellular,jul,thu,...,1,999,0,nonexistent,1.4,93.918,-42.7,4.958,5228.1,no
4115,39,admin.,married,high.school,no,yes,no,telephone,jul,fri,...,1,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
4116,27,student,single,high.school,no,no,no,cellular,may,mon,...,2,999,1,failure,-1.8,92.893,-46.2,1.354,5099.1,no
4117,58,admin.,married,high.school,no,no,no,cellular,aug,fri,...,1,999,0,nonexistent,1.4,93.444,-36.1,4.966,5228.1,no


In [24]:
print(data.shape)
data.info()
for col in data.columns:
    print(f'{col}: {data[col].unique()}\n')

(4119, 21)
<class 'pandas.DataFrame'>
RangeIndex: 4119 entries, 0 to 4118
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             4119 non-null   int64  
 1   job             4119 non-null   str    
 2   marital         4119 non-null   str    
 3   education       4119 non-null   str    
 4   default         4119 non-null   str    
 5   housing         4119 non-null   str    
 6   loan            4119 non-null   str    
 7   contact         4119 non-null   str    
 8   month           4119 non-null   str    
 9   day_of_week     4119 non-null   str    
 10  duration        4119 non-null   int64  
 11  campaign        4119 non-null   int64  
 12  pdays           4119 non-null   int64  
 13  previous        4119 non-null   int64  
 14  poutcome        4119 non-null   str    
 15  emp.var.rate    4119 non-null   float64
 16  cons.price.idx  4119 non-null   float64
 17  cons.conf.idx   4119 non-null   f

In [26]:
#set  unknown to NaN
data = data.replace('unknown', pd.NA)
data.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 4119 entries, 0 to 4118
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             4119 non-null   int64  
 1   job             4080 non-null   str    
 2   marital         4108 non-null   str    
 3   education       3952 non-null   str    
 4   default         3316 non-null   str    
 5   housing         4014 non-null   str    
 6   loan            4014 non-null   str    
 7   contact         4119 non-null   str    
 8   month           4119 non-null   str    
 9   day_of_week     4119 non-null   str    
 10  duration        4119 non-null   int64  
 11  campaign        4119 non-null   int64  
 12  pdays           4119 non-null   int64  
 13  previous        4119 non-null   int64  
 14  poutcome        4119 non-null   str    
 15  emp.var.rate    4119 non-null   float64
 16  cons.price.idx  4119 non-null   float64
 17  cons.conf.idx   4119 non-null   float64
 18 

age                 0
job                39
marital            11
education         167
default           803
housing           105
loan              105
contact             0
month               0
day_of_week         0
duration            0
campaign            0
pdays               0
previous            0
poutcome            0
emp.var.rate        0
cons.price.idx      0
cons.conf.idx       0
euribor3m           0
nr.employed         0
y                   0
dtype: int64

**Structure of the dataset**

The data contains 4,119 observations, 20 input features and 1 target variable. 10 are numerical and 9 are catecorgical variables. There are three types of variables (integer, float and string) which can be seen from the code output above.

**Missing Values**

No explicit NaN values were present in the raw data. However, several categorical columns contained the string "unknown", representing implicit missing values. These were replaced with NaN. One exception was made for `poutcome`, which contains "nonexistent". This was kept as a valid category since it carries a clear meaning: the client was never contacted in a previous campaign. It is not missing information. The most affected columns are default (803 missing, ~19.5%), education (167, ~4.1%), housing and loan (105 each, ~2.5%), marital (11, ~0.3%), and job (39, ~0.9%). The high missingness in default is particularly noteworthy and will require a careful handling decision

.

